# Backgammon and Doubling

Concept of "doubling" refers to doubling the stakes of the game at backgammon. It is represented by a die like cube with faces: 2, 4, 8, 16, 32, 64. At the start cube sits in the middle at value 1, owned by no one. Before your roll you may **double**: offer to play on for twice the stakes. Your opponent must then **take** — accept, after which the cube (now showing 2) sits on their side and only they may double next — or **drop**, conceding the current stake and ending the game.

Before every dice roll, player should consider doubling the stakes. Essentially calculating their current equity as best as they can by their game sense.

At a high level, doubling the stakes of the game is like saying:
- "I am at an advantage here. YIELD!!"
- "if you like to continue playing this game, I would like to double the stakes"

Then the opponent needs to consider their position and decide whether to take this challenge or yield as suggested.

This beautiful game theoric concept adds a new dimension of testing the wits of players. On top of playing the best move to maximize equity, a more precise understanding of the position is required. 

## Equity and the Take Point

Equity is how often player wins or loses a game. 

It could be for example: 70-30 or 40-60 for a match with no gammons. OR S:60,G:10-S:40,G:2 -> single game win chance of 60% and gammon chance of 10%

Initially let's run an equity calculation without considering gammons.
- **Drop:** you lose 1 point, for certain.
- **Take:** you play on for 2 points. Winning with probability $p$, your equity is $2p - 2(1 - p) = 4p - 2$.


Take when taking beats dropping: $4p - 2 \ge -1$, so $p \ge \tfrac{1}{4}$. This is the **25% take point**: accept any double you win at least a quarter of the time. That is the dead-cube number. Once the cube can come back your way, owning it is worth enough to pull the take point down to about **20%**.

The doubler's mirror of the take point is the **cash point**. Once the opponent likes their side less than their take point — you at about 75%, 80% with a live cube — you double them out: they pass and you bank a point. Between 50% and the cash point sits the **doubling window**: good enough to double, not so good they must drop.

## Gammons bend the line

A **gammon** is a double-stake win. Threaten one and doubling can backfire: the opponent drops and you bank one point when playing on might have won two. That is **too good to double**.

Gammons move every threshold. With a dead cube they fold into one number. Your cubeless equity is $E = 2W - 1 + G$, with $W$ your win chance and $G$ your gammon chance, and $m = (E + 1)/2 = W + G/2$ is the same equity on a 0-to-1 scale; with no gammons it is just $W$. Double past $m = 0.5$, the opponent drops past $m = 0.75$, too good past $m = 1$.

Each cell of the map is a position: your win chance down the side, your gammon chance across the top. A gammon is a win too, so it never exceeds the win chance. Colour is the cube action in a money game, straight from the rule.

- **Yellow** — double, and the opponent has a **take**.
- **Blue** — double, and the opponent must **drop**; you cash a point.

Grey is the two ends: *no double* along the bottom, where you do not like it enough yet, and *too good* in the top right, where you would rather play on for the gammon than cash. The lines are dead-cube; a live cube nudges the drop boundary out a touch.

In [1]:
import plotly.graph_objects as go

WINS = list(range(40, 100, 5))
GAMS = list(range(0, 46, 5))


def action(win, gammon):
    w, g = win / 100, gammon / 100
    if g > w:
        return None
    m = w + g / 2
    if m < 0.5:
        return 0
    if m <= 0.75:
        return 1
    if m <= 1.0:
        return 2
    return 3


LABELS = ["no double", "double / take", "double / drop", "too good"]
CELL = {0: "", 1: "take", 2: "drop", 3: "too good", None: ""}
z = [[action(w, g) for g in GAMS] for w in WINS]
text = [[CELL[action(w, g)] for g in GAMS] for w in WINS]

colors = ["#eef0f2", "#ffd94d", "#4da3ff", "#c2c2c2"]
scale = []
for i, c in enumerate(colors):
    scale += [[i / 4, c], [(i + 1) / 4, c]]

fig = go.Figure(go.Heatmap(
    z=z, x=[f"{g}%" for g in GAMS], y=[f"{w}%" for w in WINS],
    text=text, texttemplate="%{text}", textfont=dict(size=11),
    colorscale=scale, zmin=-0.5, zmax=3.5, xgap=1, ygap=1,
    colorbar=dict(tickvals=[0, 1, 2, 3], ticktext=LABELS, title="action")))
fig.update_layout(
    title="Money game: double / take / drop by win and gammon chance",
    xaxis_title="gammon-win chance", yaxis_title="game-win chance",
    margin=dict(t=60))
fig.show()

## Match play: factoring in the score

In a money game the take point sat at 25% because a point was always worth a point. A match breaks that. You are racing to a target, so a point is worth more or less depending on the score: the point that takes you from 6 to 7 wins the match, the point that takes you from 2 to 3 barely moves anything. Once a point's worth floats with the score, so does the take point.

What a point is worth at a given score is your chance of winning the match from there. A **match equity table** (MET) is that chance written out for every score, indexed by how many points each side is *away* from the target. The map below uses the classic Woolsey–Heinrich MET for a 7-point match.

The take-versus-drop comparison is the same as in money; only the currency changed, from points to match-winning chances. Dropping locks in one score. Taking splits you between winning and losing the doubled game, each landing at a different score. Read those three chances off the MET,

$$\text{TP} = \frac{a - b}{c - b}, \qquad \begin{aligned} a &= \text{drop (doubler scores 1)} \\ b &= \text{take and lose 2} \\ c &= \text{take and win 2} \end{aligned}$$

and the take point is the win rate where taking and dropping break even.

Set the two sliders to how far the giver and the taker are from 7; the crosshair marks the cell, and its number is the taker's take point. Far from the target the score barely bites: at 5-away each it is the money 25%. At 3-away each it is **30%**. At 2-away each it is 30% again, and the cube dies on the take, because two points end the match and there is no redouble to lean on. Sideways the score pulls harder. A taker 4-away against a giver 2-away needs **34%**, because losing the doubled game ends the match. The mirror, 2-away against 4-away, takes at **20%**, because losing only levels the score and winning ends it.

Two edges the map leaves off. At 1-away the Crawford rule turns the cube off for a game, and at 1-away each the cube means nothing: take everything and play for the win. And these take points are cubeless; where the cube stays live, like 5-away each, a redouble you would own pulls them a little below what the map shows.

In [2]:
import plotly.graph_objects as go

MET_PCT = [
    [50, 70, 75, 83, 85, 90, 91, 94, 95, 97, 97, 98, 98, 99, 99],
    [30, 50, 60, 68, 75, 81, 85, 88, 91, 93, 94, 95, 96, 97, 98],
    [25, 40, 50, 59, 66, 71, 76, 80, 84, 87, 90, 92, 94, 95, 96],
    [17, 32, 41, 50, 58, 64, 70, 75, 79, 83, 86, 88, 90, 92, 93],
    [15, 25, 34, 42, 50, 57, 63, 68, 73, 77, 81, 84, 87, 89, 90],
    [10, 19, 29, 36, 43, 50, 56, 62, 67, 72, 76, 79, 82, 85, 87],
    [9, 15, 24, 30, 37, 44, 50, 56, 61, 66, 70, 74, 78, 81, 84],
    [6, 12, 20, 25, 32, 38, 44, 50, 55, 60, 65, 69, 73, 77, 80],
    [5, 9, 16, 21, 27, 33, 39, 45, 50, 55, 60, 64, 68, 72, 76],
    [3, 7, 13, 17, 23, 28, 34, 40, 45, 50, 55, 60, 64, 68, 71],
    [3, 6, 10, 14, 19, 24, 30, 35, 40, 45, 50, 55, 59, 63, 67],
    [2, 5, 8, 12, 16, 21, 26, 31, 36, 40, 45, 50, 54, 58, 62],
    [2, 4, 6, 10, 13, 18, 22, 27, 32, 36, 41, 46, 50, 54, 58],
    [1, 3, 5, 8, 11, 15, 19, 23, 28, 32, 37, 42, 46, 50, 54],
    [1, 2, 4, 7, 10, 13, 16, 20, 24, 29, 33, 38, 42, 46, 50],
]
MATCH = 7


def met(t, d):
    if t <= 0:
        return 100.0
    if d <= 0:
        return 0.0
    return float(MET_PCT[t - 1][d - 1])


def take_point(taker_score, giver_score):
    t, d = MATCH - taker_score, MATCH - giver_score
    a = met(t, d - 1)
    b = met(t, d - 2)
    c = met(t - 2, d)
    if c == b:
        return None
    return (a - b) / (c - b) * 100


scores = list(range(0, 6))
n = len(scores)
z = [[take_point(t, d) for t in scores] for d in scores]
text = [[f"{v:.0f}" if v is not None else "" for v in row] for row in z]

fig = go.Figure(go.Heatmap(
    z=z, x=[f"{MATCH - s}-away" for s in scores], y=[f"{MATCH - s}-away" for s in scores],
    text=text, texttemplate="%{text}%", textfont=dict(size=11),
    colorscale="YlGnBu", xgap=1, ygap=1, colorbar=dict(title="take point")))

lo, hi = -0.5, n - 0.5
start = n - 1
fig.update_layout(shapes=[
    dict(type="rect", xref="x", yref="y", x0=lo, x1=hi, y0=start - 0.5, y1=start + 0.5,
         fillcolor="rgba(0,0,0,0.20)", line=dict(width=0)),
    dict(type="rect", xref="x", yref="y", x0=start - 0.5, x1=start + 0.5, y0=lo, y1=hi,
         fillcolor="rgba(0,0,0,0.20)", line=dict(width=0)),
])

giver_steps = [dict(method="relayout", label=f"{MATCH - s}-away",
                    args=[{"shapes[0].y0": i - 0.5, "shapes[0].y1": i + 0.5}])
               for i, s in enumerate(scores)]
taker_steps = [dict(method="relayout", label=f"{MATCH - s}-away",
                    args=[{"shapes[1].x0": j - 0.5, "shapes[1].x1": j + 0.5}])
               for j, s in enumerate(scores)]

fig.update_layout(
    sliders=[
        dict(active=start, y=-0.02, len=0.6, currentvalue={"prefix": "cube giver = "},
             steps=giver_steps),
        dict(active=start, y=-0.32, len=0.6, currentvalue={"prefix": "cube taker = "},
             steps=taker_steps),
    ],
    title="Match-play take point by points away (7-point match, Woolsey MET)",
    xaxis_title="cube taker — points away",
    yaxis_title="cube giver — points away",
    margin=dict(t=60, b=140))
fig.show()

## Scenarios: one shot, read at several scores

At the board the win chance is often a shot count. Every roll is one of 36, equally likely, so the counts are small enough to carry in your head, and each position below is one of them. Set the count against the line: 25% for money, and whatever the map says in a match.

- **You need one specific number** — a 6 to jump the prime, or the one open point to come in against a five-point board. Eleven of 36 rolls contain it: **30.6%**. Entering from the bar is the same count from the other end: $1 - (n/6)^2$ against $n$ closed points, 30.6% at five, 75% at three.
- **Their blot is open to a direct shot.** A blot one pip away is hit by 11 of 36 rolls, six pips away by **17 of 36**, because combinations start reaching it too. Two blots, 4 and 6 away, are the union of the two sets, **27 of 36**, not the sum. Seen from the other side of the table, you are at 53% against the single shot and at **25%** against the double.
- **They have two swings at one number.** Missing it once is $25/36$, missing it twice is $(25/36)^2$, so they hit **51.8%** of the time and you are at 48.2%.
- **You need doubles**, to jump a prime or to take two off and save the gammon: 6 of 36, **16.7%**. An indirect shot, 7 or more pips away, is at best 6 of 36 too, and 2 of 36 at eleven.
- **You are closed out**: 0%.

The table reads each of these against the map. Rows are the shot counts, columns the scores, and every cell is take or drop; a take that clears the line by less than a point is marked bare.

In [3]:
from IPython.display import HTML

MONEY_DEAD, MONEY_LIVE = 25.0, 20.0
SHOTS = [
    ("one number: 11/36", 11 / 36),
    ("direct shot six away, against you: 1 - 17/36", 1 - 17 / 36),
    ("two swings at one number, against you: (25/36)^2", (25 / 36) ** 2),
    ("double shot, against you: 1 - 27/36", 1 - 27 / 36),
    ("doubles only: 6/36", 6 / 36),
    ("closed out", 0.0),
]
SCORES = [
    ("money, dead cube", MONEY_DEAD),
    ("money, live cube", MONEY_LIVE),
    ("5-away each", take_point(2, 2)),
    ("3-away each", take_point(4, 4)),
    ("2-away each", take_point(5, 5)),
    ("4-away vs 2-away", take_point(3, 5)),
    ("2-away vs 4-away", take_point(5, 3)),
]


def verdict(win, line):
    margin = win * 100 - line
    if margin < 0:
        return "drop"
    return "bare take" if margin < 1 else "take"


header = "".join(f"<th>{name}<br><small>line {line:.0f}%</small></th>" for name, line in SCORES)
rows = "".join(
    f"<tr><td>{name}<br><small>you win {win * 100:.1f}%</small></td>"
    + "".join(f"<td>{verdict(win, line)}</td>" for _, line in SCORES)
    + "</tr>"
    for name, win in SHOTS
)
HTML(f"<table><thead><tr><th>position</th>{header}</tr></thead><tbody>{rows}</tbody></table>")

position,"money, dead cubeline 25%","money, live cubeline 20%",5-away eachline 25%,3-away eachline 30%,2-away eachline 30%,4-away vs 2-awayline 34%,2-away vs 4-awayline 20%
one number: 11/36you win 30.6%,take,take,take,bare take,bare take,drop,take
"direct shot six away, against you: 1 - 17/36you win 52.8%",take,take,take,take,take,take,take
"two swings at one number, against you: (25/36)^2you win 48.2%",take,take,take,take,take,take,take
"double shot, against you: 1 - 27/36you win 25.0%",bare take,take,bare take,drop,drop,drop,take
doubles only: 6/36you win 16.7%,drop,drop,drop,drop,drop,drop,drop
closed outyou win 0.0%,drop,drop,drop,drop,drop,drop,drop


Read the first row. For money the line is 25% and 30.6% is a comfortable take. At 5-away each the line is still 25%: same position, same take. At 3-away each the line is 30% and the same shot is a bare take, clearing by half a point. Trailing 4-away against 2-away it is a drop. Leading 2-away against 4-away it is an easy take. Nothing about the checkers changed and the dice did not change; the score moved the line under a fixed shot.

The double-shot row is the sharpest. Against you it is exactly the money line, a take with a live cube, a drop at 3-away each, and a take again when you lead. The last two rows never come close to any line on the map, and the two-swings row never falls below one.

The race has no shot to count. It is pips, the average roll moves 8.17 of them, and its take points are a different table and a different post.

The helpers below build every count on this page from the 36 rolls, so you can check any distance or any board yourself.

In [4]:
DICE = [(a, b) for a in range(1, 7) for b in range(1, 7)]


def reachable(a, b):
    return {a, 2 * a, 3 * a, 4 * a} if a == b else {a, b, a + b}


def shots(distance):
    return sum(1 for a, b in DICE if distance in reachable(a, b))


def hit_prob(distance):
    return shots(distance) / 36


def enter_prob(points_closed):
    return 1 - (points_closed / 6) ** 2


def shots_any(*distances):
    wanted = set(distances)
    return sum(1 for a, b in DICE if reachable(a, b) & wanted)


def over_rolls(prob, rolls):
    return 1 - (1 - prob) ** rolls


for k in list(range(1, 13)) + [15, 16, 18, 20, 24]:
    print(f"hit {k:>2} pips away: {shots(k):>2}/36 = {hit_prob(k) * 100:4.1f}%")
print()
for closed in range(1, 7):
    print(f"enter vs {closed} closed points: {enter_prob(closed) * 100:5.1f}%")
print()
doubles = sum(1 for a, b in DICE if a == b)
miss = sum(1 for a, b in DICE if a != 3 and b != 3)
pips = sum(4 * a if a == b else a + b for a, b in DICE)
print(f"any double: {doubles}/36 = {doubles / 36 * 100:.1f}%")
print(f"miss a given number: {miss}/36 = {miss / 36 * 100:.1f}%")
print(f"average pips per roll: {pips / 36:.2f}")
print()
for pair in [(4, 6), (3, 6), (1, 6), (2, 5)]:
    n = shots_any(*pair)
    print(f"double shot at {pair[0]} and {pair[1]}: {n}/36 = {n / 36 * 100:.1f}%")
print()
one = enter_prob(5)
for r in range(1, 4):
    print(f"one specific number within {r} roll(s): {over_rolls(one, r) * 100:.1f}%")

hit  1 pips away: 11/36 = 30.6%
hit  2 pips away: 12/36 = 33.3%
hit  3 pips away: 14/36 = 38.9%
hit  4 pips away: 15/36 = 41.7%
hit  5 pips away: 15/36 = 41.7%
hit  6 pips away: 17/36 = 47.2%
hit  7 pips away:  6/36 = 16.7%
hit  8 pips away:  6/36 = 16.7%
hit  9 pips away:  5/36 = 13.9%
hit 10 pips away:  3/36 =  8.3%
hit 11 pips away:  2/36 =  5.6%
hit 12 pips away:  3/36 =  8.3%
hit 15 pips away:  1/36 =  2.8%
hit 16 pips away:  1/36 =  2.8%
hit 18 pips away:  1/36 =  2.8%
hit 20 pips away:  1/36 =  2.8%
hit 24 pips away:  1/36 =  2.8%

enter vs 1 closed points:  97.2%
enter vs 2 closed points:  88.9%
enter vs 3 closed points:  75.0%
enter vs 4 closed points:  55.6%
enter vs 5 closed points:  30.6%
enter vs 6 closed points:   0.0%

any double: 6/36 = 16.7%
miss a given number: 25/36 = 69.4%
average pips per roll: 8.17

double shot at 4 and 6: 27/36 = 75.0%
double shot at 3 and 6: 28/36 = 77.8%
double shot at 1 and 6: 24/36 = 66.7%
double shot at 2 and 5: 23/36 = 63.9%

one specific n

## Conclusion

A cube decision is one comparison, and the line it is made against was built in three steps. For money with a dead cube the line is 25%. Gammons fold into one number, $m = W + G/2$, and the line becomes three: double at 0.5, cash at 0.75, too good at 1. In a match the line moves with every point scored, because a point is worth whatever it does to your chance of reaching the target, and the match equity table turns that into a take point for every pair of scores.

The scenarios show what that means at the board. A shot count is fixed by the dice: 30.6% for one number, 25% for a double shot against you, 48.2% against two swings, 16.7% for doubles. What the count is worth is set by how many points each side is away. So when the cube comes: count the shot first, then read the scoreboard to find out what the shot is worth.

**[In-browser notebook →](/lite/notebooks/index.html?path=2026-09-24-backgammon-cube-and-dice.ipynb)**